[![Open In Colab](/_static/colab-badge.svg)](https://colab.research.google.com/github/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-generation/Using_BoltzGen.ipynb)
[![Get Notebook](/_static/get-notebook-badge.svg)](https://raw.githubusercontent.com/OpenProteinAI/openprotein-docs/refs/heads/main/source/python-api/structure-generation/Using_BoltzGen.ipynb)
[![View In GitHub](/_static/view-in-github-badge.svg)](https://github.com/OpenProteinAI/openprotein-docs/blob/main/source/python-api/structure-generation/Using_BoltzGen.ipynb)

# Using BoltzGen
This tutorial shows you how to use the BoltzGen model to design novel
protein structures.

The examples here are mainly using those from the [original
documentation](https://github.com/HannesStark/boltzgen) but
adapted to show how it can be run using the OpenProtein platform, which
can then be combined with our other workflows!

Full credit for the examples and model go to the authors of
boltzgen!

## Unconditional monomer design

The basic execution of BoltzGen would be an unconditional design of a
protein structure of a certain length. You would need 3 things:

1.  An authenticated OpenProtein session
2.  Length of the protein
3.  Number of designs `N` desired

In [1]:
import openprotein
session = openprotein.connect()
length = 150
N = 3

In [2]:
boltzgen = session.models.boltzgen
boltzgen.generate?

Signature:
boltzgen.generate(
    query: str | bytes | openprotein.molecules.protein.Protein | openprotein.molecules.complex.Complex | openprotein.prompt.models.Query | None = None,
    design_spec: openprotein.models.foundation.boltzgen_schema.BoltzGenDesignSpec | dict[str, typing.Any] | None = None,
    structure_file: str | bytes | typing.BinaryIO | None = None,
    N: int = 1,
    diffusion_batch_size: int | None = None,
    step_scale: float | None = None,
    noise_scale: float | None = None,
    scaffolds: dict[str, str | bytes | typing.BinaryIO] | None = None,
    scaffold_set: openprotein.scaffolds.Scaffolds | str | None = None,
    extra_structure_files: dict[str, str | bytes | typing.BinaryIO] | None = None,
    **kwargs,
) -> openprotein.models.foundation.boltzgen.BoltzGenFuture
Docstring:
Run a protein structure generate job using BoltzGen.

Parameters
----------
query : str or bytes or Protein or Complex or Query, optional
    A query representing the design specification

To generate designs, we can use our convenient `Query` interface. 

Alternatively, our python interface also supports the official design specifications from BoltzGen too. Look at the Appendix for an example.

In [3]:
from openprotein.molecules import Protein

unconditional_monomer = Protein.from_expr(length)
print("sequence:", unconditional_monomer.sequence)
print("structure mask:", unconditional_monomer.get_structure_mask())

sequence: b'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'
structure mask: [ True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  True  True  True  True  True  True  True  True  True  True
  True  True  Tru

Run the design using BoltzGen:

In [4]:
unconditional_design_job = boltzgen.generate(N=N, query=unconditional_monomer)
unconditional_design_job

BoltzGenJob(job_id='612017a6-cf64-4642-a100-eb55167c645d', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2026, 1, 17, 13, 20, 26, 44950, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

Wait for the job to finish running with `wait_until_done`.

In [5]:
unconditional_design_job.wait_until_done(verbose=True, timeout=600)

Waiting: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [03:18<00:00,  1.98s/it, status=SUCCESS]


True

Retrieve the designs as a list of `N` [Complex](../api-reference/molecules.rst#openprotein.molecules.Complex) objects. `Complex` objects represent multimers, and can hold multiple protein (and other) chains. For now, our design will only return a single chain. Let's look at the first one.

In [6]:
from molviewspec import create_builder

def display_structure(structure_string):
    builder = create_builder()
    structure = builder.download(url="mystructure.cif")\
        .parse(format="mmcif")\
        .model_structure()\
        .component()\
        .representation()\
        .color_from_source(schema="atom",
                            category_name="atom_site",
                            field_name="auth_asym_id",
                            palette={"kind": "categorical", # color by chain
                                    "colors": ["blue", "red", "green", "orange"], 
                                    "mode": "ordinal"}
                          )
    return builder.molstar_notebook(data={'mystructure.cif': structure_string}, width=500, height=400)
    
unconditional_design = unconditional_design_job.get()[0]
display_structure(unconditional_design.to_string())

<IPython.core.display.Javascript object>

## Vanilla Protein Binding

One of the basic examples in BoltzGen is to do a vanilla protein binding. To do so, we will first retrieve the structure file and parse it as a [Protein](../api-reference/molecules.rst#openprotein.molecules.Protein). We can then craft a molecular [Complex](../api-reference/molecules.rst#openprotein.molecules.Complex) with an additional chain to be designed alongside our first chain.

In [7]:
import requests
import yaml
import json
from openprotein.molecules import Complex

example_cif_string = requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_protein/1g13.cif").text
example_target = Protein.from_string(example_cif_string, format="cif", chain_id="A")
binder_query = example_target & "80"

print("target sequence:", binder_query.get_protein("A").sequence)
print("binder sequence:", binder_query.get_protein("B").sequence)
print("target structure mask:", binder_query.get_protein("A").get_structure_mask())
print("binder structure mask:", binder_query.get_protein("B").get_structure_mask())

target sequence: b'SSFSWDNCDEGKDPAVIRSLTLEPDPIIVPGNVTLSVMGSTSVPLSSPLKVDLVLEKEVAGLWIKIPCTDYIGSCTFEHFCDVLDMLIPTGEPCPEPLRTYGLPCHCPFKEGTYSLPKSEFVVPDLELPSWLTTGNYRIESVLSSSGKRLGCIKIAASLKGI'
binder sequence: b'XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX'
target structure mask: [False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False

Now we can run the example:

In [8]:
vanilla_protein_design_job = boltzgen.generate(
    query=binder_query,
    N=1,
)
vanilla_protein_design_job

BoltzGenJob(job_id='4efcf6c7-ee98-4a17-a6a7-b4d1f0e60f0f', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2026, 1, 17, 13, 32, 17, 327108, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

In [9]:
vanilla_protein_design_job.wait_until_done(verbose=True, timeout=600)

Waiting: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [01:43<00:00,  1.04s/it, status=SUCCESS]


True

Display the target + binder. Take note that chain B is the target from chain A above, and chain A is the designed binder.

In [10]:
vanilla_protein_design = vanilla_protein_design_job.get()[0]
display_structure(vanilla_protein_design.to_string())

<IPython.core.display.Javascript object>

### Vanilla Peptide with Target Binding Site

Let's run the other example which involves designing a peptide binder. We can retrieve the structure from the official example, then set the target binding sites.

In [11]:
from openprotein.molecules import Binding

example_cif_string = requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_peptide_with_target_binding_site/5cqg.cif").text
example_target = Protein.from_string(example_cif_string, format="cif", chain_id="A")
example_target.set_binding_at([343,344,251], Binding.BINDING)
peptide_binder_query = example_target & "15"

print("target sequence:", peptide_binder_query.get_protein("A").sequence)
print("binder sequence:", peptide_binder_query.get_protein("B").sequence)
print("target structure mask:", peptide_binder_query.get_protein("A").get_structure_mask())
print("binder structure mask:", peptide_binder_query.get_protein("B").get_structure_mask())

# Display our target
display_structure(example_target.to_string())

target sequence: b'MVHYYRLSLKSRQKAPKIVNSKYNSILNIALKNFRLCKKHKTKKPVQILALLQEIIPKSYFGTTTNLKRFYKVVEKILTQSSFECIHLSVLHKCYDYDAIPWLQNVEPNLRPKLLLKHNLFLLDNIVKPIIAFYYKPIKTLNGHEIKFIRKEEYISFESKVFHKLKKMKYLVEVQDEVKPRGVLNIIPKQDNFRAIVSIFPDSARKPFFKLLTSKIYKVLEEKYKTSGSLYTCWSEFTQKTQGQIYGIKVDIRDAYGNVKIPVLCKLIQSIPTHLLDSEKKNFIVDHISNQFVAFRRKIYKWNHGLLQGDPLSGCLCELYMAFMDRLYFSNLDKDAFIHRTVDDYFFCSPHPHKVYDFELLIKGVYQVNPTKTRTNLPTHRHPQDEIPYCGKIFNLTTRQVRTLYKLPPNYEIRHKFKLWNFNNQISDDNPARFLQKAMDFPFICNSFTKFEFNTVFNDQRTVFANFYDAMICVAYKFDAAMMALRTSFLVNDFGFIWLVLSSTVRAYASRAFKKIVTYKGGKYRKVTFQCLKSIAWRAFLAVLKRRTEIYKGLIDRIKSREKLTMKFHDGEVDASYFCKLPEKFRFVKINRKASI'
binder sequence: b'XXXXXXXXXXXXXXX'
target structure mask: [False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False False False False False False False False
 False False False False False F

<IPython.core.display.Javascript object>

In [12]:
vanilla_peptide_design_job = boltzgen.generate(
    query=peptide_binder_query,
    N=1,
)
vanilla_peptide_design_job

BoltzGenJob(job_id='6fee8571-cae6-4ffc-bbe7-23ec5a49dddc', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2026, 1, 17, 13, 40, 7, 797809, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)

Wait for and retrieve the result. Display the output design.

In [13]:
vanilla_peptide_design_job.wait_until_done(verbose=True, timeout=900)
vanilla_peptide_design = vanilla_peptide_design_job.get()[0]
display_structure(vanilla_peptide_design.to_string())

Waiting: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████| 100/100 [02:53<00:00,  1.74s/it, status=SUCCESS]


<IPython.core.display.Javascript object>

# Next Steps

You can run more of the examples from the BoltzGen repository. Take note that any command line arguments to `boltzgen run` can be passed as `kwargs` to the `boltzgen.design` function.

You can also move on to the next step of the design pipeline by running inverse folding using PoET-2. Refer to the walkthrough of [Inverse Folding with PoET-2](../../walkthroughs/PoET-2_inverse_folding.ipynb) for an example.

# Appendix

## Using the BoltzGen design specification

To support any non-standard workflows that may not be fully covered by our [Query](../api-reference/prompt.rst#openprotein.prompt.Query) interface, our Python interface also fully supports the raw official BoltzGen specification. This can be used by supplying them directly as `design_spec`. 

However, when doing so, it will usually be necessary to also provide additional structure files to accompany the design specification. These can be provided using `extra_structure_files` as a mapping of filenames to files.

The following is an example of running the vanilla protein design job with this interface.

In [ ]:
import io

design_spec = yaml.safe_load(requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_protein/1g13prot.yaml").text)
structure_file = requests.get("https://raw.githubusercontent.com/HannesStark/boltzgen/refs/heads/main/example/vanilla_protein/1g13.cif").text

vanilla_protein_design_job_ = boltzgen.generate(
    design_spec=design_spec,
    extra_structure_files={"1g13.cif": io.BytesIO(structure_file.encode())},
    N=1,
)
vanilla_protein_design_job_

BoltzGenJob(job_id='8146f068-ce59-43d3-9aec-782cce54099e', job_type='/models/boltzgen', status=<JobStatus.PENDING: 'PENDING'>, created_date=datetime.datetime(2026, 1, 17, 14, 2, 7, 381776, tzinfo=TzInfo(0)), start_date=None, end_date=None, prerequisite_job_id=None, progress_message=None, progress_counter=0, sequence_length=None)